# GDELT (BigQuery) → extraction locale (raw)

Ce notebook exécute une requête BigQuery sur la table publique GDELT et sauvegarde le résultat dans `data/raw/`.

## Auth BigQuery (dans l'ordre)
- `GOOGLE_APPLICATION_CREDENTIALS` (chemin vers un JSON Service Account)
- `credentials/credentials.json` à la racine du repo
- sinon ADC (Application Default Credentials)

In [19]:
from __future__ import annotations
import logging
import sys
sys.path.append(str(Path().resolve().parent))
from pathlib import Path

import scripts.data_pipeline as dp

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s - %(message)s")
sys.path.append(str(Path().resolve().parent))

PROJECT_ROOT = Path().resolve().parent
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)


In [20]:
import importlib
importlib.reload(dp)

<module 'scripts.data_pipeline' from 'E:\\junior\\Formations\\Hackathons\\Hackathon_iSHEEROXDatacamp\\scripts\\data_pipeline.py'>

In [21]:
query = """
SELECT *
FROM `gdelt-bq.gdeltv2.events`
WHERE ActionGeo_CountryCode = 'BN'
  AND YEAR >= 2025
LIMIT 1000
""".strip()

client = dp.get_bq_client()
df = dp.extract_raw_data(client, query)

raw_dir = PROJECT_ROOT / "data" / "raw"
raw_dir.mkdir(parents=True, exist_ok=True)
dp.load_raw_data(df, raw_dir / "gdelt_bn_2025.csv")
df.head()

2026-04-29 17:46:10,258 INFO scripts.data_pipeline - Using service account credentials from E:\junior\Formations\Hackathons\Hackathon_iSHEEROXDatacamp\credentials\credentials.json (project=hackathon-datacampxisheero)
2026-04-29 17:46:10,266 INFO scripts.data_pipeline - Running query (104 chars)
2026-04-29 17:46:26,509 INFO scripts.data_pipeline - DataFrame saved to E:\junior\Formations\Hackathons\Hackathon_iSHEEROXDatacamp\data\raw\gdelt_bn_2025.csv


,GLOBALEVENTID,SQLDATE,MonthYear,Year,FractionDate,Actor1Code,Actor1Name,Actor1CountryCode,Actor1KnownGroupCode,Actor1EthnicCode,...,ActionGeo_Type,ActionGeo_FullName,ActionGeo_CountryCode,ActionGeo_ADM1Code,ActionGeo_ADM2Code,ActionGeo_Lat,ActionGeo_Long,ActionGeo_FeatureID,DATEADDED,SOURCEURL
0,1295504124,20260322,202603,2026,2026.2247,BEN,COTONOU,BEN,NaN,NaN,...,4,"Godomey, Atlantique, Benin",BN,BN09,5876,6.38948,2.34581,-1333714,20260322074500,https://english.news.cn/africa/20260322/476f31...
1,1295504126,20260322,202603,2026,2026.2247,BEN,BENIN,BEN,NaN,NaN,...,4,"Godomey, Atlantique, Benin",BN,BN09,5876,6.38948,2.34581,-1333714,20260322074500,https://english.news.cn/africa/20260322/476f31...
2,1295514475,20260322,202603,2026,2026.2247,BEN,BENIN,BEN,NaN,NaN,...,1,Benin,BN,BN,NaN,9.50000,2.25000,BN,20260322094500,https://fr.allafrica.com/stories/202603220031....
3,1295487488,20260322,202603,2026,2026.2247,HLH,HOSPITAL,NaN,NaN,NaN,...,1,Benin,BN,BN,NaN,9.50000,2.25000,BN,20260322021500,https://fr.allafrica.com/stories/202603220011....
4,1295486907,20260322,202603,2026,2026.2247,BEN,BENINESE,BEN,NaN,NaN,...,1,Benin,BN,BN,NaN,9.50000,2.25000,BN,20260322023000,https://allafrica.com/stories/202603220016.html


In [15]:
job = client.query(query)

print(job.state)
print(job.error_result)

DONE
{'reason': 'quotaExceeded', 'location': 'unbilled.analysis', 'message': 'Quota exceeded: Your project exceeded quota for free query bytes scanned. For more information, see https://cloud.google.com/bigquery/docs/troubleshoot-quotas'}
